In [ ]:
import polars as pl
import pandas as pd

In [ ]:
df = (
    pl.scan_csv("data/rating_complete.csv")
    .collect()
)

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df = pd.read_csv("data/rating_complete.csv")
df.shape

In [ ]:
"""import pandas as pd

from surprise import Dataset
from surprise import Reader
from surprise import SVD
from surprise.model_selection import train_test_split
from surprise.accuracy import rmse, mae

# Keep only required columns
ratings = df[['user_id', 'anime_id', 'rating']]

reader = Reader(rating_scale=(ratings.rating.min(), ratings.rating.max()))

data = Dataset.load_from_df(
    ratings[['user_id', 'anime_id', 'rating']],
    reader
)

trainset, testset = train_test_split(
    data,
    test_size=0.2,
    random_state=42
)

model = SVD(
    n_factors=100,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42
)

model.fit(trainset)

predictions = model.test(testset)

print("RMSE:", rmse(predictions))
print("MAE :", mae(predictions))"""

### BENCHMARK

In [ ]:
"""from surprise.accuracy import rmse, mae

rmse_value = rmse(predictions, verbose=False)
mae_value = mae(predictions, verbose=False)

rating_range = ratings["rating"].max() - ratings["rating"].min()

rmse_pct = (rmse_value / rating_range) * 100
mae_pct = (mae_value / rating_range) * 100

print(f"RMSE: {rmse_value:.3f} ({rmse_pct:.2f}%)")
print(f"MAE : {mae_value:.3f} ({mae_pct:.2f}%)")"""

In [ ]:
animes = pd.read_csv("data/anime.csv")

In [ ]:
animes.shape

In [ ]:
animes.head()

In [ ]:
animes_filtered = animes[['MAL_ID',  'Name',  'Genres',  'Rating',  'Studios', 'Type', 'Premiered', 'Source', 'Members']].rename(columns={
    'MAL_ID': 'anime_id',
    'Rating': 'Age_rating'
    })


In [ ]:
df.rename(columns={'rating': 'score'}, inplace=True)

In [ ]:
animes_filtered.head()

In [ ]:
df.head()

In [ ]:
df_total = pd.merge(df, animes_filtered, on='anime_id', how='left')

In [ ]:
df_total.shape

In [ ]:
df_total.head() 

In [ ]:
df_total.columns

In [ ]:
df = df_total.copy()

In [ ]:
import numpy as np
import pandas as pd

from scipy.sparse import hstack, csr_matrix

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# ============================================================
# PARAMETERS
# ============================================================

N_HASH_FEATURES = 128

# ============================================================
# NUMERIC FEATURES
# ============================================================

numeric = csr_matrix(
    df[["score", "Members"]]
    .fillna(0)
    .astype(np.float32)
)

# ============================================================
# HASH GENRES
# ============================================================

genre_hasher = HashingVectorizer(
    n_features=N_HASH_FEATURES,
    binary=True,
    alternate_sign=False,
    token_pattern=r'[^,]+'
)

genre_matrix = genre_hasher.transform(
    df["Genres"].fillna("")
)

# ============================================================
# OTHER CATEGORICAL VARIABLES
# ============================================================

categorical_cols = [
    "Type",
    "Age_rating",
    "Source"
]

encoder = OneHotEncoder(
    handle_unknown="ignore"
)

cat_matrix = encoder.fit_transform(
    df[categorical_cols].fillna("Unknown")
)

# ============================================================
# INTERACTION MATRIX
# ============================================================

X = hstack([
    numeric,
    genre_matrix,
    cat_matrix
]).tocsr()

# ============================================================
# AGGREGATE PER USER
# ============================================================

user_codes, unique_users = pd.factorize(df["user_id"])

n_users = len(unique_users)

aggregation = csr_matrix(
    (
        np.ones(len(df), dtype=np.float32),
        (user_codes, np.arange(len(df)))
    )
)

X_user = aggregation @ X

# Average instead of sum
counts = np.asarray(aggregation.sum(axis=1)).ravel()
counts[counts == 0] = 1

X_user = X_user.multiply(1 / counts[:, None])

# ============================================================
# SCALE
# ============================================================

scaler = StandardScaler(with_mean=False)

X_scaled = scaler.fit_transform(X_user)

from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import numpy as np

# ============================================================
# FIND THE BEST NUMBER OF CLUSTERS
# ============================================================

SAMPLE_SIZE = min(10000, X_scaled.shape[0])

rng = np.random.default_rng(42)
sample_idx = rng.choice(
    X_scaled.shape[0],
    SAMPLE_SIZE,
    replace=False
)

X_sample = X_scaled[sample_idx]

k_values = range(2, 21)

scores = []

for k in k_values:

    model = MiniBatchKMeans(
        n_clusters=k,
        batch_size=10000,
        random_state=42,
        n_init="auto"
    )

    labels = model.fit_predict(X_sample)

    score = silhouette_score(
        X_sample,
        labels,
        metric="euclidean"
    )

    scores.append(score)

    print(f"k = {k:2d} | Silhouette = {score:.4f}")

best_k = k_values[np.argmax(scores)]

print(f"\nBest k = {best_k}")

plt.figure(figsize=(8,5))

plt.plot(k_values, scores, marker="o")
plt.xticks(range(2, 21, 3))
plt.xlabel("Quantidade de Clusters (k)")
plt.ylabel("Score")
plt.title("Teste de Silhuette")

plt.grid(alpha=0.3)

plt.show()

# Use the best value found
N_CLUSTERS = best_k

In [ ]:
#N_CLUSTERS = 8 

# ============================================================
# CLUSTER
# ============================================================

kmeans = MiniBatchKMeans(
    n_clusters=N_CLUSTERS,
    batch_size=10000,
    random_state=42,
    n_init="auto"
)

clusters = kmeans.fit_predict(X_scaled)

# ============================================================
# RESULTS
# ============================================================

users_cluster = pd.DataFrame({
    "user_id": unique_users,
    "Cluster": clusters
})

print(users_cluster.head())

# ============================================================
# PCA VISUALIZATION
# ============================================================

pca = PCA(n_components=2, random_state=42)

X_plot = pca.fit_transform(X_scaled.toarray())

plt.figure(figsize=(10,8))

plt.scatter(
    X_plot[:,0],
    X_plot[:,1],
    c=clusters,
    cmap="tab10",
    s=10,
    alpha=0.6
)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("User Clusters")
plt.colorbar(label="Cluster")
plt.show()

In [ ]:
from collections import defaultdict, Counter
import matplotlib.pyplot as plt
import numpy as np

# ============================================================
# Attach cluster to every interaction
# ============================================================

df = df.merge(users_cluster, on="user_id", how="left")

# ============================================================
# Count genres per cluster (streaming, no explode)
# ============================================================

cluster_genres = defaultdict(Counter)

for cluster, genres in zip(df["Cluster"], df["Genres"]):

    if pd.isna(genres):
        continue

    for genre in genres.split(","):
        cluster_genres[cluster][genre.strip()] += 1

# ============================================================
# Create cluster names
# ============================================================

cluster_names = {}

for cluster in sorted(cluster_genres.keys()):

    top3 = [
        genre
        for genre, _
        in cluster_genres[cluster].most_common(3)
    ]

    cluster_names[cluster] = " / ".join(top3)

# ============================================================
# Plot top genres for each cluster
# ============================================================

n_clusters = len(cluster_names)

fig, axes = plt.subplots(
    n_clusters,
    1,
    figsize=(5, 2*n_clusters)
)

if n_clusters == 1:
    axes = [axes]

for ax, cluster in zip(axes, sorted(cluster_names.keys())):

    top10 = cluster_genres[cluster].most_common(5)

    genres = [g for g, _ in top10]
    counts = [c for _, c in top10]

    ax.barh(genres, counts)

    ax.invert_yaxis()

    ax.set_title(
        f"Cluster {cluster}: {cluster_names[cluster]}"
    )

    ax.set_xlabel("Ocorrências")

plt.tight_layout()
plt.show()

# ============================================================
# PCA with cluster names
# ============================================================

plt.figure(figsize=(7, 5))

for cluster in sorted(cluster_names.keys()):

    idx = clusters == cluster

    plt.scatter(
        X_plot[idx, 0],
        X_plot[idx, 1],
        s=10,
        alpha=0.5,
        label=cluster_names[cluster]
    )

plt.title("Distribuição dos Clusters de Usuários")
plt.legend(
    title="Cluster",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

# ============================================================
# Print summary
# ============================================================

print("\nCluster Summary\n")

for cluster in sorted(cluster_names.keys()):

    print(f"Cluster {cluster}")
    print(f"Name: {cluster_names[cluster]}")

    for genre, count in cluster_genres[cluster].most_common(10):
        print(f"   {genre:<20} {count:,}")

    print()